<a href="https://colab.research.google.com/github/hexa-box/colab/blob/main/Exemple_LLM_et_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# Téléchargement du Modèle LLM
# =====================================================================
PATH_MODELE = "qwen2.5-1.5b-instruct-q4_k_m.gguf"

import os
if not os.path.exists(PATH_MODELE):
    print(f"Downloading {PATH_MODELE}...")
    # Note: The model name was qwen2.5-1.5b-instruct-q4_k_m.gguf, but the available model on HuggingFace is qwen1_5-1_8b-chat-q4_k_m.gguf. Adjusting URL and assuming this is the intended model.
    !wget https://huggingface.co/Qwen/Qwen1.5-1.8B-Chat-GGUF/resolve/main/qwen1_5-1_8b-chat-q4_k_m.gguf -O {PATH_MODELE}
else:
    print(f"Model {PATH_MODELE} already exists.")

--2026-06-23 19:31:32--  https://huggingface.co/Qwen/Qwen1.5-1.8B-Chat-GGUF/resolve/main/qwen1_5-1_8b-chat-q4_k_m.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.23, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/65be2ae3c1a44b6ef120fc1a/7c6663a0337434fc7dacb27eef36d9536d0c59a44ce549ea6e08a86ea141f48f?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27qwen1_5-1_8b-chat-q4_k_m.gguf%3B+filename%3D%22qwen1_5-1_8b-chat-q4_k_m.gguf%22%3B&Expires=1782246692&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViZTJhZTNjMWE0NGI2ZWYxMjBmYzFhLzdjNjY2M2EwMzM3NDM0ZmM3ZGFjYjI3ZWVmMzZkOTUzNmQwYzU5YTQ0Y2U1NDllYTZlMDhhODZlYTE0MWY0OGZcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQit

In [ ]:
# =====================================================================
# Installation des dépendances
# =====================================================================
!pip install -qqq numpy==1.26.4 # Downgrade numpy to a compatible version
!pip install -qqq chromadb==0.4.22
!pip install -qqq llama_cpp_python==0.2.37

# IMPORTANT: After running this cell, please restart the Colab runtime (Runtime -> Restart runtime)
# This is necessary to ensure all libraries correctly pick up the downgraded NumPy version.

import chromadb
from llama_cpp import Llama

# =====================================================================
# PARTIE 1 : Initialisation du Modèle LLM
# =====================================================================
PATH_MODELE = "qwen2.5-1.5b-instruct-q4_k_m.gguf"

print("1/5 -> Chargement du modèle Qwen...")
# Setting verbose=True to bypass the UnsupportedOperation: fileno error in Colab
llm = Llama(model_path=PATH_MODELE, embedding=True, n_ctx=2048, verbose=True)


# =====================================================================
# PARTIE 2 : Initialisation de la Base de Données Vectorielle (ChromaDB)
# =====================================================================
# On crée une base de données éphémère (en mémoire vive).
# Dans un vrai projet, on pourrait la sauvegarder sur le disque dur.
print("2/5 -> Initialisation de ChromaDB (Base vectorielle)...")
chroma_client = chromadb.Client()

# On crée une "collection" (l'équivalent d'une table dans une base de données)
collection = chroma_client.create_collection(name="ma_base_de_connaissances")


# =====================================================================
# PARTIE 3 : Remplissage de la Base de Données
# =====================================================================
base_documents = [
    "Le protocole de sécurité 'Alpha-4' exige la fermeture des portes en moins de 30 secondes.",
    "Le chat de garde du bureau s'appelle Pixel et il adore les croquettes au saumon.",
    "La clé de l'armoire informatique se trouve cachée dans le pot de fleurs de l'accueil.",
    "La réunion d'équipe a lieu tous les jeudis matin à 9h30 en salle de conférence B."
]

print("3/5 -> Indexation des documents dans ChromaDB...")

# Pour chaque document, on génère son embedding (son vecteur) et on l'ajoute à Chroma
for i, doc in enumerate(base_documents):
    # 1. On transforme le texte en vecteur grâce à Qwen
    reponse_embedding = llm.create_embedding(doc)
    vecteur = reponse_embedding["data"][0]["embedding"]

    # 2. On ajoute le tout dans ChromaDB
    collection.add(
        embeddings=[vecteur],  # Les chiffres mathématiques
        documents=[doc],       # Le texte d'origine
        ids=[f"id_{i}"]        # Un identifiant unique obligatoire
    )


# =====================================================================
# PARTIE 4 : Requête de l'utilisateur et RECHERCHE (Retrieval)
# =====================================================================
question_utilisateur = "Où est cachée la clé de l'armoire informatique ?"
print(f"\nQuestion posée : '{question_utilisateur}'")

print("4/5 -> [RAG - Retrieval] ChromaDB cherche le document le plus pertinent...")

# 1. On transforme la question de l'utilisateur en vecteur
reponse_embedding_question = llm.create_embedding(question_utilisateur)
vecteur_question = reponse_embedding_question["data"][0]["embedding"]

# 2. On demande à ChromaDB de trouver le document le plus proche (n_results=1)
resultat_recherche = collection.query(
    query_embeddings=[vecteur_question],
    n_results=1
)

# On extrait le texte du document trouvé par ChromaDB
contexte_trouve = resultat_recherche["documents"][0][0]
print(f"-> Document trouvé automatiquement par ChromaDB : '{contexte_trouve}'")


# =====================================================================
# PARTIE 5 : Génération de la réponse par le LLM
# =====================================================================
print("5/5 -> [RAG - Generation] Le LLM rédige la réponse finale...")

messages_du_chat = [
    {
        "role": "system",
        "content": "Réponds à la question en utilisant le contexte fourni. Sois précis."
    },
    {
        "role": "user",
        "content": f"Contexte : {contexte_trouve}\n\nQuestion : {question_utilisateur}"
    }
]

reponse_modele = llm.create_chat_completion(
    messages=messages_du_chat,
    temperature=0.1,
    max_tokens=100
)

texte_final = reponse_modele["choices"][0]["message"]["content"]

# Affichage du résultat
print("\n" + "="*40)
print("RÉSULTAT DU RAG (AVEC CHROMADB)")
print("="*40)
print(f"Réponse de Qwen : {texte_final}")
print("="*40)

1/5 -> Chargement du modèle Qwen...


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 
Model metadata: {'general.file_type': '15', 'tokenizer.ggml.bos_token_id': '151643', 'general.architecture': 'qwen2', 'general.name': 'Qwen1.5-1.8B-Chat-AWQ-fp16', 'qwen2.block_count': '24', 'qwen2.context_length': '32768', 'tokenizer.chat_template': "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}", 'qwen2.attention.head_count_kv': '16', 'tokenizer.ggml.padding_token_id': '151643', 'qwen2.embedding_length': '2048', 'qwen2.attention.layer_norm_rms_epsilon': '0.000001', 'qwen2.attention.head_count': '16', 'tokenizer.ggml.eos_token_id': '151645', 'qwen2.rope.freq_base': '1000000.000000', 'qwen2.use_parallel_residual': 'true', 'general.q

2/5 -> Initialisation de ChromaDB (Base vectorielle)...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


3/5 -> Indexation des documents dans ChromaDB...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given



Question posée : 'Où est cachée la clé de l'armoire informatique ?'
4/5 -> [RAG - Retrieval] ChromaDB cherche le document le plus pertinent...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Llama.generate: prefix-match hit


-> Document trouvé automatiquement par ChromaDB : 'La clé de l'armoire informatique se trouve cachée dans le pot de fleurs de l'accueil.'
5/5 -> [RAG - Generation] Le LLM rédige la réponse finale...

RÉSULTAT DU RAG (AVEC CHROMADB)
Réponse de Qwen : La clé de l'armoire informatique est cachée dans le pot de fleurs de l'accueil. Le pot de fleurs est un endroit où les visiteurs peuvent trouver des fleurs pour améliorer leur environnement et la qualité de vie en général. Il peut être trouvé dans différents endroits, tels que dans les salons, les boutiques de produits alimentaires, les restaurants, les musées et les parcs naturels.

Dans le cas où


# New Section